<a href="https://colab.research.google.com/github/anangshachatterjee/anangsha-chatterjee-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule

I will rank content pages for refresh review using two decision-time signals:

1. **Staleness:** pages that have gone a long time since their last update receive a higher priority.
2. **CTR opportunity:** pages with meaningful search visibility, a relatively strong search position, but low CTR receive a higher priority for CTR-focused review.

The rule is intentionally simple and transparent. It is a baseline for comparison with the ML model in Week 5, not a claim that these signals cause traffic decline.

### Reason codes

- `STALE_VISIBLE_PAGE` — the page has been unupdated for a long time and has meaningful impressions.
- `LOW_CTR_VISIBLE_PAGE` — the page has meaningful impressions, is positioned within the first 20 results, but has low CTR.
- `STALE_AND_LOW_CTR` — both conditions are present.
- `GENERAL_REVIEW` — neither specific condition is met.

### Action labels

- `refresh` — prioritize a content refresh review.
- `refresh_and_review_ctr` — prioritize a refresh with special attention to search-result CTR.
- `monitor` — lower-priority review rather than immediate action.

The score uses only information available at the decision moment. The decline label and product-generated flags are used only for signal checking/evaluation, not as score inputs.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Public starter dataset
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
# ---------------------------------------------------------
# Signal 1: STALENESS
# Signal 2: CTR relative to search position
#
# is_declining is used ONLY to evaluate the signal.
# It is NOT used in the baseline score.
# ---------------------------------------------------------

# Make sure the outcome is boolean
df["is_declining_label"] = df["is_declining_label"].astype(bool)

# -----------------------------
# Signal 1: Staleness
# -----------------------------
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 89, 179, np.inf],
    labels=["<90 days", "90-179 days", "180+ days"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

staleness_check["declining_rate"] = (
    staleness_check["declining_rate"] * 100
).round(2)

print("SIGNAL 1 — STALENESS")
display(staleness_check)


# -----------------------------
# Signal 2: CTR + position
# -----------------------------
df["ctr_position_bucket"] = np.select(
    [
        (df["avg_position"] > 0) &
        (df["avg_position"] <= 20) &
        (df["ctr"] < 0.5),

        (df["avg_position"] > 0) &
        (df["avg_position"] <= 20) &
        (df["ctr"] >= 0.5),

        (df["avg_position"] > 20)
    ],
    [
        "Top-20 + low CTR",
        "Top-20 + higher CTR",
        "Position >20"
    ],
    default="No usable position"
)

ctr_position_check = (
    df.groupby("ctr_position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

ctr_position_check["declining_rate"] = (
    ctr_position_check["declining_rate"] * 100
).round(2)

print("\nSIGNAL 2 — CTR VS POSITION")
display(ctr_position_check)

KeyError: 'is_declining_label'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.